In [10]:
import gymnasium as gym
import numpy as np
import wandb
from collections import defaultdict
from itertools import count

In [11]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [12]:
CONFIG = {
    "env_name": "FrozenLake-v1",
    "test_episodes": 20,
    "learning_rate": 0.1,
    "gamma": 0.99,
    "target_reward": 0.80
}

In [13]:
class QLearningAgent:
    def __init__(self, env):
        self.env = env
        self.q_table = defaultdict(float)
        self.state, _ = self.env.reset()

    def sample_env(self):
        """Perform one step in the environment and return the transition."""
        state = self.state
        # Simple epsilon-greedy exploration
        action = self.env.action_space.sample() if np.random.random() < 0.1 else self.best_action(state)
        
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        
        # Store transition to return
        transition = (state, action, reward, next_state)
        
        # Reset if finished
        if terminated or truncated:
            self.state, _ = self.env.reset()
        else:
            self.state = next_state
            
        return transition

    def best_action(self, state):
        """Returns the action with the highest Q-value for a state."""
        values = [self.q_table[(state, a)] for a in range(self.env.action_space.n)]
        return np.argmax(values)

    def value_update(self, s, a, r, next_s):
        """Classic Q-Learning update rule."""
        best_next_q = max([self.q_table[(next_s, a)] for a in range(self.env.action_space.n)])
        
        # Bellman Equation
        self.q_table[(s, a)] += CONFIG["learning_rate"] * (
            r + CONFIG["gamma"] * best_next_q - self.q_table[(s, a)]
        )

    def play_episode(self, env):
        """Plays one full episode greedily for evaluation."""
        state, _ = env.reset()
        total_reward = 0
        done = False
        while not done:
            action = self.best_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
        return total_reward

In [14]:
with wandb.init(project="frozen-lake-rl", config=CONFIG) as run:
        # Setup environments
        train_env = gym.make(CONFIG["env_name"], is_slippery=True)
        test_env = gym.make(CONFIG["env_name"], is_slippery=True)
        
        agent = QLearningAgent(train_env)
        best_reward = 0.0

        print(f"Training on {CONFIG['env_name']}...")

        # 2. Modern Training Loop
        for iter_no in count(1):
            # Step A: Interact and Update
            s, a, r, next_s = agent.sample_env()
            agent.value_update(s, a, r, next_s)

            # Step B: Evaluate periodically (every 100 steps to save time)
            if iter_no % 100 == 0:
                avg_reward = sum(agent.play_episode(test_env) for _ in range(CONFIG["test_episodes"])) / CONFIG["test_episodes"]
                
                # 3. Log to wandb
                wandb.log({"reward": avg_reward, "iter": iter_no})

                if avg_reward > best_reward:
                    print(f"Iter {iter_no}: New Best Reward {avg_reward:.3f}")
                    best_reward = avg_reward
                    run.summary["best_reward"] = best_reward

                if avg_reward > CONFIG["target_reward"]:
                    print(f"✅ Solved in {iter_no} iterations!")
                    break

Training on FrozenLake-v1...
Iter 132900: New Best Reward 0.050
Iter 133400: New Best Reward 0.100
Iter 134100: New Best Reward 0.150
Iter 134400: New Best Reward 0.200
Iter 137000: New Best Reward 0.300
Iter 138800: New Best Reward 0.350
Iter 141000: New Best Reward 0.550
Iter 141600: New Best Reward 0.600
Iter 144000: New Best Reward 0.650
Iter 146800: New Best Reward 0.700
Iter 147000: New Best Reward 0.900
✅ Solved in 147000 iterations!


iter,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
reward,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▂█▅▃
best_reward,0.9
iter,147000
reward,0.9
